Used to upload the datasets into Google Colab for cleaning

In [ ]:
from google.colab import files
uploaded = files.upload()

To confirm whether or not the csv's have been uploaded properly one of the .csv's has been outputted

In [ ]:
import pandas as pd

orders = pd.read_csv("orders.csv")
orders.head()

,order_id,customer_id,service_type,order_created_at,promised_window_hours,pickup_zone,dropoff_zone,priority_level,order_value,booking_channel,special_handling_flag
0,O00001,C0292,Passenger,2024-08-20 14:43:00,6,Airport,South,Medium,126.65,App,0
1,O00002,C0459,Passenger,2024-05-14 22:16:00,24,North,AIRPORT,Low,109.30,App,0
2,O00003,C0161,Passenger,2025-09-02 14:37:00,4,West,AIRPORT,High,33.50,Phone,0
3,O00004,C0520,Parcel,2025-01-11 17:15:00,2,RiverSide,North,Medium,10.04,App,1
4,O00005,C0558,Retail,2025-02-17 19:32:00,12,Riverside,SOUTH,Low,125.58,Phone,0


Then proceeding to read all of the .csv files to make them available for further cleaning/processing.

In [ ]:
import pandas as pd

orders = pd.read_csv("orders.csv")
deliveries = pd.read_csv("deliveries.csv")
complaints = pd.read_csv("complaints.csv")
incidents = pd.read_csv("incidents.csv")
customers = pd.read_csv("customers.csv")
drivers = pd.read_csv("drivers.csv")
vehicles = pd.read_csv("vehicles.csv")
hubs = pd.read_csv("hubs.csv")

In [ ]:
orders.shape
deliveries.shape

(950, 13)

To understand the data structures and to identify any missing data ___.info() was used to have a better view of what is available.

In [ ]:
orders.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1250 entries, 0 to 1249
Data columns (total 11 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   order_id               1250 non-null   object 
 1   customer_id            1250 non-null   object 
 2   service_type           1250 non-null   object 
 3   order_created_at       1250 non-null   object 
 4   promised_window_hours  1250 non-null   int64  
 5   pickup_zone            1250 non-null   object 
 6   dropoff_zone           1250 non-null   object 
 7   priority_level         1250 non-null   object 
 8   order_value            1250 non-null   float64
 9   booking_channel        1225 non-null   object 
 10  special_handling_flag  1250 non-null   int64  
dtypes: float64(1), int64(2), object(8)
memory usage: 107.6+ KB


In [ ]:
deliveries.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 950 entries, 0 to 949
Data columns (total 13 columns):
 #   Column                         Non-Null Count  Dtype  
---  ------                         --------------  -----  
 0   delivery_id                    950 non-null    object 
 1   order_id                       950 non-null    object 
 2   driver_id                      950 non-null    object 
 3   vehicle_id                     950 non-null    object 
 4   hub_id                         950 non-null    object 
 5   dispatch_time                  950 non-null    object 
 6   delivery_completed_at          931 non-null    object 
 7   delivery_status                950 non-null    object 
 8   route_distance_km              950 non-null    float64
 9   manual_route_override_count    950 non-null    int64  
 10  proof_of_completion_missing    950 non-null    int64  
 11  customer_rating_post_delivery  936 non-null    float64
 12  fuel_or_charge_cost            950 non-null    flo

Proceeding to drop any duplicates within the datasets, fixing date columns, and filling out any missing values found.

In [ ]:
orders = orders.drop_duplicates()
deliveries = deliveries.drop_duplicates()

In [ ]:
orders['order_created_at'] = pd.to_datetime(orders['order_created_at'], errors='coerce')

In [ ]:
deliveries['customer_rating_post_delivery'] = deliveries['customer_rating_post_delivery'].fillna(0)
deliveries['manual_route_override_count'] = deliveries['manual_route_override_count'].fillna(0)

Then the text is cleaned by using the following code.

In [ ]:
orders['service_type'] = orders['service_type'].str.lower().str.strip()

By merging the datasets, a bigger picture comes together to understand the information

In [ ]:
df = orders.merge(deliveries, on='order_id', how='left')
df = df.merge(customers, on='customer_id', how='left')
df = df.merge(drivers, on='driver_id', how='left')
df = df.merge(vehicles, on='vehicle_id', how='left')
df = df.merge(hubs, on='hub_id', how='left')

In [ ]:
df.head()
df.shape

(1250, 49)

Then columns are made for complaints, incidents, and low ratings to have a better picture of the problems NorthStar is facing.

In [ ]:
df['has_complaint'] = df['order_id'].isin(complaints['order_id']).astype(int)
df['has_incident'] = df['delivery_id'].isin(incidents['delivery_id']).astype(int)
df['low_rating'] = (df['customer_rating_post_delivery'] <= 2).astype(int)
df['high_override'] = (df['manual_route_override_count'] > 2).astype(int)

Then adding the cleaned data into a .csv and downloading to make a clean dataset

In [ ]:
df.to_csv("cleaned_data.csv", index=False)

In [ ]:
from google.colab import files
files.download("cleaned_data.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

# Business Analysis

Comparing the complaints to operational issues received

In [ ]:
df.groupby('has_complaint')[['has_incident', 'high_override', 'low_rating']].mean()

,has_incident,high_override,low_rating
has_complaint,,,
0,0.208290,0.072539,0.047668
1,0.164912,0.063158,0.031579


To find out what is the cause of low rating:

In [ ]:
df.groupby('low_rating')[['has_incident', 'high_override', 'has_complaint']].mean()

,has_incident,high_override,has_complaint
low_rating,,,
0,0.195816,0.070293,0.230962
1,0.254545,0.072727,0.163636


Now to find out the hubs performances:

In [ ]:
df.groupby('hub_id')[['has_complaint', 'has_incident', 'customer_rating_post_delivery']].mean()

,has_complaint,has_incident,customer_rating_post_delivery
hub_id,,,
H01,0.191176,0.220588,3.812353
H02,0.150943,0.283019,3.913679
H03,0.260504,0.268908,3.797647
H04,0.204724,0.259843,3.884646
H05,0.243478,0.295652,3.605739
H06,0.201923,0.240385,3.844808
H07,0.260870,0.278261,3.814348
H08,0.242188,0.250000,3.793516


To find out cost compared to performance:

In [ ]:
df.groupby('hub_id')[['fuel_or_charge_cost', 'customer_rating_post_delivery']].mean()

,fuel_or_charge_cost,customer_rating_post_delivery
hub_id,,
H01,12.755809,3.812353
H02,12.565000,3.913679
H03,12.744202,3.797647
H04,13.167008,3.884646
H05,13.686000,3.605739
H06,13.319231,3.844808
H07,12.922087,3.814348
H08,11.708203,3.793516


What service types receive the most complaints:

In [ ]:
df.groupby('service_type')[['has_complaint', 'customer_rating_post_delivery']].mean()

,has_complaint,customer_rating_post_delivery
service_type,,
business,0.206061,3.816746
medical,0.237410,3.837685
parcel,0.224026,3.833783
passenger,0.225806,3.771870
retail,0.242424,3.803393
